# __*Ebrahim Part*__

## __*PDF Generation*__

### __*Importing Libraries*__

In [20]:
import shutil
from docx import Document
from datetime import datetime
from docx.shared import Pt
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT

### __*Function for Capitalizing Words for Display*__

In [21]:
def capitalizeWords(s):
    return " ".join(word.capitalize() for word in s.split(" "))

### __*Function for Formatting Numbers for Display*__

In [22]:
def formatNumber(num):
    if num > 1000:
        return round(num)  # Round to the nearest integer
    elif num > 100:
        return f"{num:.1f}"  # Show 1 decimal place
    else:
        return f"{num:.2f}"  # Show 2 decimal places

### __*Function for Calculating Percentage*__

In [23]:
def calculatePercentage(value, total):
    if total == 0:
        return "0%"  # Avoid division by zero
    percentage = (value / total) * 100
    return f"{percentage:.0f}%" if percentage.is_integer() else f"{percentage:.1f}%"

### __*Current Date Generation*__

In [24]:
current_date = datetime.today().strftime("%d-%m-%Y")

### __*Function for Replacing Placeholders in Word Doc While Preserving Format*__

In [26]:
# Function to replace text while handling split runs
def replaceText(doc, old_text, new_text):
    for para in doc.paragraphs:
        full_text = "".join(run.text for run in para.runs)  # Combine all runs
        if old_text in full_text:
            full_text = full_text.replace(old_text, new_text)  # Replace placeholder
            
            # Clear existing runs
            for run in para.runs:
                run.text = ""

            # Reapply the modified text in the first run (keeps formatting)
            para.runs[0].text = full_text  

    # Also check inside tables (if placeholders are inside tables)
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for para in cell.paragraphs:
                    full_text = "".join(run.text for run in para.runs)
                    if old_text in full_text:
                        full_text = full_text.replace(old_text, new_text)
                        for run in para.runs:
                            run.text = ""
                        para.runs[0].text = full_text

### __*Function to Add Asset Details Table in Word Doc*__

In [27]:
def addAssetDetailsTable(doc,asset_details):
    for para in doc.paragraphs:
        if "Asset Details" in para.text:
            insert_paragraph = para
            break
    
    # Insert a table with 2 columns and (n+1) rows (including header)
    table = doc.add_table(rows=len(asset_details), cols=2)
    
    # Apply formatting to table
    table.style = 'Table Grid'
    
    for row_idx, (key, value) in enumerate(asset_details.items()):
        # Left column (bold)
        left_cell = table.cell(row_idx, 0)
        left_run = left_cell.paragraphs[0].add_run(key)
        left_run.bold = True
        left_run.font.name = "Times New Roman"
        left_run.font.size = Pt(14)
    
        # Right column (normal text)
        right_cell = table.cell(row_idx, 1)
        right_run = right_cell.paragraphs[0].add_run(str(value))
        right_run.font.name = "Times New Roman"
        right_run.font.size = Pt(14)
    
        # Center align text
        for cell in [left_cell, right_cell]:
            cell.paragraphs[0].alignment = 1  # Center alignment
    
    # Move table below "Asset Details"
    insert_paragraph.insert_paragraph_before("\n")  # Add space before table
    insert_paragraph._element.addnext(table._element)  # Insert table after "Asset Details"

### __*Function to Set Vertical Alignment of Cell to Center*__

In [28]:
def setVerticalAlignment(cell):
    """Set vertical alignment to center."""
    tc = cell._element
    tcPr = tc.find(qn('w:tcPr')) or OxmlElement('w:tcPr')
    vAlign = OxmlElement('w:vAlign')
    vAlign.set(qn('w:val'), "center")
    tcPr.append(vAlign)
    tc.append(tcPr)

### __*Function to Add Heir Details Table in Word Doc*__

In [29]:
def addHeirDetailTable(doc,heir_details):
    for para in doc.paragraphs:
        if "Heir Details" in para.text:
            heir_paragraph = para
            break
    
    # Define table headers
    headers = ["Count", "Heir Relation", "Category", "Shares", "Shares Amount",
               "Share of Each Person", "Share Amount of Each Person"]
    
    # Insert table with headers + rows for heir details
    heir_table = doc.add_table(rows=1 + len(heir_details), cols=len(headers))
    heir_table.style = 'Table Grid'
    
    # Format header row
    for col_idx, header in enumerate(headers):
        cell = heir_table.cell(0, col_idx)
        run = cell.paragraphs[0].add_run(header)
        run.bold = True
        run.font.name = "Times New Roman"
        run.font.size = Pt(14)
        cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER  # Horizontal center
        setVerticalAlignment(cell)  # Vertical center
    
    # Fill table rows
    for row_idx, heir in enumerate(heir_details, start=1):
        for col_idx, key in enumerate(headers):
            cell = heir_table.cell(row_idx, col_idx)
            run = cell.paragraphs[0].add_run(str(heir[key]))
            run.font.name = "Times New Roman"
            run.font.size = Pt(14)
            cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER  # Horizontal center
            setVerticalAlignment(cell)  # Vertical center
    
    # Move table below "Heir Details"
    heir_paragraph.insert_paragraph_before("\n")  # Add space before table
    heir_paragraph._element.addnext(heir_table._element)  # Insert table after "Heir Details

## __*Inheritance Calculation*__

### __*Mapping Dictionaries*__

In [30]:
relation_map = {
    "son": "num_sons",
    "daughter": "num_daughters",
    "father": "has_father",
    "mother": "has_mother",
    "husband": "has_husband",
    "wife": "num_wives",
    "son's son": "num_sons_sons",
    "son's daughter": "num_sons_daughters",
    "father's father": "has_fathers_father",
    "father's mother": "has_fathers_mother",
    "mother's mother": "has_mothers_mother",
    "real brother": "num_real_brothers",
    "real sister": "num_real_sisters",
    "paternal brother": "num_paternal_brothers",
    "paternal sister": "num_paternal_sisters",
    "maternal sibling": "num_maternal_siblings",
    "son's son's son": "num_sons_sons_sons",
    "son's son's daughter": "num_sons_sons_daughters"
}

share_map = {
    "HusbandShare": "husband",
    "PerWifeShare": "wife",
    "PerSonShare": "son",
    "PerDaughterShare": "daughter",
    "PerSonsDaughterShare": "son's daughter",
    "FatherShare": "father",
    "PerSonsSonsDaughterShare": "son's son's daughter",
    "MotherShare": "mother",
    "FathersFatherShare": "father's father",
    "FathersMotherShare": "father's mother",
    "MothersMotherShare": "mother's mother",
    "PerRealSisterShare": "real sister",
    "PerPaternalSisterShare": "paternal sister",
    "PerMaternalSiblingShare": "maternal sibling",
    "PerSonsSonShare": "son's son",
    "PerSonsSonsSonShare": "son's son's son",
    "PerRealBrotherShare": "real brother",
    "PerPaternalBrotherShare": "paternal brother"
}

### __*Dictionary for Relation Counts*__

In [31]:
relation_counts = {
    "num_wives": 0,
    "num_sons": 0,
    "num_daughters": 0,
    "num_sons_sons": 0,
    "num_sons_daughters": 0,
    "has_father": 0,
    "num_sons_sons_daughters": 0,
    "num_sons_sons_sons": 0,
    "has_mother": 0,
    "num_real_brothers": 0,
    "num_real_sisters": 0,
    "num_paternal_brothers": 0,
    "num_paternal_sisters": 0,
    "num_maternal_siblings": 0,
    "has_fathers_father": 0,
    "has_fathers_mother": 0,
    "has_mothers_mother": 0,
    "has_husband": 0
}

### __*Function for Relation Counts Calcuation*__

In [32]:
def relationCountsCalculation(heir_list):
    temp_relation_counts = relation_counts
    for heir in heir_list:
        relation = heir.get("relation")
        val = heir.get("val", 0)
        temp_relation_counts[relation_map[relation]] = val
    return temp_relation_counts

### __*Function for Assigning Shares to Each Heir*__

In [33]:
def assignHeirShares(shares,heir_list):
    for heir in heir_list:
        relation = heir.get("relation")
        heir["amount"] = shares.get(next((key for key, value in share_map.items() if value == relation), ""), 0)

## __*API Routes*__

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyswip import Prolog

# Creating Flask Server App
app = Flask(__name__)

# Enables Cross Origin Resources Sharing in Flask App
CORS(app)

# Load Prolog Rules from the rule.pl file
prolog = Prolog()
prolog.consult("rules.pl")

# Defining API Route for Inheritance Calculation
@app.route('/inheritance-calculator-2', methods=['POST'])
def inheritance_calculator_2():

    # Extracting Received Data
    data = request.get_json()

    # Exceptional Handling
    if not data:
        return jsonify({"error": "Invalid JSON input"}), 400
    
    total_wealth = data.get('total_amount', 0)  # Extracting Total Amount
    heir_list = data.get('heir_list', [])  # Extracting Heir List

    # Testing
    print("--------------------------------------| RECEIVED INHERITANCE CALCULATION REQUEST FROM CLIENT |--------------------------------------")
    print("Received Data :",)
    print("Total Wealth : ",total_wealth)
    print("Hier List :")
    for heir in heir_list:
        print(heir)

    # Calculating Relation Counts
    relation_counts = relationCountsCalculation(heir_list)
    
    # Querying Prolog for Inheritance Shares
    result = list(prolog.query(
        f"inheritance_calculator({total_wealth}, {relation_counts['has_husband']}, {relation_counts['num_wives']}, "
        f"{relation_counts['num_sons']}, {relation_counts['num_daughters']}, {relation_counts['num_sons_sons']}, "
        f"{relation_counts['num_sons_daughters']}, {relation_counts['num_sons_sons_daughters']}, {relation_counts['num_sons_sons_sons']}, "
        f"{relation_counts['has_father']}, {relation_counts['has_mother']}, {relation_counts['num_real_brothers']}, "
        f"{relation_counts['num_real_sisters']}, {relation_counts['num_paternal_brothers']}, {relation_counts['num_paternal_sisters']}, "
        f"{relation_counts['num_maternal_siblings']}, {relation_counts['has_fathers_father']}, {relation_counts['has_fathers_mother']}, "
        f"{relation_counts['has_mothers_mother']}, HusbandShare, PerWifeShare, PerSonShare, PerDaughterShare, PerSonsDaughterShare, "
        f"FatherShare, PerSonsSonsDaughterShare, MotherShare, FathersFatherShare, FathersMotherShare, MothersMotherShare, "
        f"PerRealSisterShare, PerPaternalSisterShare, PerMaternalSiblingShare, PerSonsSonShare, PerSonsSonsSonShare, "
        f"PerRealBrotherShare, PerPaternalBrotherShare)"
    ))

    # Assigning Shares to Heirs
    if result:
        shares = result[0]
        assignHeirShares(shares,heir_list)

    # Testing
    print()
    print("Response Data :")     
    for heir in heir_list:
        print(heir)
        
    return jsonify({
        "status": "success",
        "heir_list": heir_list,
    })

# Defining API Route for PDF Generation
@app.route('/inheritance-calculation-pdf', methods=['POST'])
def inheritance_calculation_pdf():
    data = request.get_json()

    if not data:
        return jsonify({"error": "Invalid JSON input"}), 400

    # Extracting values from request data
    amount = float(data.get('total_amount') or 0)
    funeral_expenses = float(data.get('funeral_expenses') or 0)
    mehr = float(data.get('mehr') or 0)
    debt = float(data.get('debt') or 0)
    will = float(data.get('will') or 0)
    heir_list = data.get('heir_list', [])


    distributable_amount = amount - funeral_expenses - mehr - debt - will

    asset_details = {
        "Total Amount": amount,
        "Funeral & Burial Expenses": funeral_expenses,
        "Haq Mehr": mehr,
        "Debt & Loans": debt,
        "Will": will,
        "Distributable Amount": distributable_amount
    }

    # Filtering only values greater than zero
    asset_details = {key: value for key, value in asset_details.items() if value}

    heir_details = [
        {
            "Count": heir.get("val"),
            "Heir Relation": capitalizeWords(heir.get("relation")),
            "Category": capitalizeWords(heir.get("category", ["", ""])[1]),
            "Shares": calculatePercentage(heir.get("amount")*heir.get("val"),distributable_amount),
            "Shares Amount": formatNumber(heir.get("amount")*heir.get("val")),
            "Share of Each Person": calculatePercentage(heir.get("amount"),distributable_amount),
            "Share Amount of Each Person": formatNumber(heir.get("amount"))
        }
        for heir in heir_list
    ]

    # Generating PDF

    # Creating Copy of Inheritance Calculation Template Word Doc

    template_file = "Inheritance-Calculation-Template.docx"
    new_file = "Modified-Inheritance-Calculation.docx"

    shutil.copy(template_file, new_file) 
    
    # Loading Document
    doc = Document(new_file)

    # Replacing Current Date as Calculation Date in Word Doc    
    replaceText(doc, "<<CalculationDate>>", current_date)

    # Adding Asset Details Table
    addAssetDetailsTable(doc,asset_details)

    # Adding Heir Details Table
    addHeirDetailTable(doc,heir_details)

    # Saving Document
    doc.save(new_file)
    print("Updated document saved as:", new_file)

    # Testing
    print()
    print("--------------------------------------| RECEIVED PDF GENERATION REQUEST FROM CLIENT |--------------------------------------")
    print("Asset Details : ")
    print(asset_details)
    print()
    print("Heir Details : ")
    for heir in heir_details:
        print(heir)

    return jsonify({"status": "success"}), 200  # ✅ Return only a success response


if __name__ == '__main__':
    app.run(debug=False, port=8080)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:8080
Press CTRL+C to quit
127.0.0.1 - - [21/Mar/2025 16:50:50] "OPTIONS /inheritance-calculator-2 HTTP/1.1" 200 -
127.0.0.1 - - [21/Mar/2025 16:50:50] "POST /inheritance-calculator-2 HTTP/1.1" 200 -


--------------------------------------| RECEIVED INHERITANCE CALCULATION REQUEST FROM CLIENT |--------------------------------------
Received Data :
Total Wealth :  8000
Hier List :
{'relation': 'son', 'category': ['common', 'children'], 'limit': 99, 'val': 2}
{'relation': 'daughter', 'category': ['common', 'children'], 'limit': 99, 'val': 1}

Response Data :
{'relation': 'son', 'category': ['common', 'children'], 'limit': 99, 'val': 2, 'amount': 3200}
{'relation': 'daughter', 'category': ['common', 'children'], 'limit': 99, 'val': 1, 'amount': 1600}


127.0.0.1 - - [21/Mar/2025 16:50:54] "OPTIONS /inheritance-calculation-pdf HTTP/1.1" 200 -
C:\Users\e-bra\AppData\Local\Temp\ipykernel_13120\806683724.py:4: FutureWarning: Truth-testing of elements was a source of confusion and will always return True in future versions. Use specific 'len(elem)' or 'elem is not None' test instead.
  tcPr = tc.find(qn('w:tcPr')) or OxmlElement('w:tcPr')
127.0.0.1 - - [21/Mar/2025 16:50:55] "POST /inheritance-calculation-pdf HTTP/1.1" 200 -


Updated document saved as: Modified-Inheritance-Calculation.docx

--------------------------------------| RECEIVED PDF GENERATION REQUEST FROM CLIENT |--------------------------------------
Asset Details : 
{'Total Amount': 10000.0, 'Funeral & Burial Expenses': 500.0, 'Haq Mehr': 500.0, 'Debt & Loans': 500.0, 'Will': 500.0, 'Distributable Amount': 8000.0}

Heir Details : 
{'Count': 2, 'Heir Relation': 'Son', 'Category': 'Children', 'Shares': '80%', 'Shares Amount': 6400, 'Share of Each Person': '40%', 'Share Amount of Each Person': 3200}
{'Count': 1, 'Heir Relation': 'Daughter', 'Category': 'Children', 'Shares': '20%', 'Shares Amount': 1600, 'Share of Each Person': '20%', 'Share Amount of Each Person': 1600}
